# ETFbook · flujos netos de ETFs

**Que responde este cuaderno**

1. Cuanto flujo neto entra o sale en el trimestre.
2. Que ETF se lleva ese flujo.
3. Si el dinero entra de forma sostenida o en un solo mes.
4. Que region domina.

**Datos**: `etf_flows_trend_analysis_2025.csv` (12 filas: 4 ETFs por 3 meses). Se descarga por URL al ejecutar.

**Como correrlo**: Runtime > Run all. Solo pandas y matplotlib.

**Escala**: es un extracto trimestral de 4 ETFs, no una serie historica completa.

In [ ]:
import io, pathlib, urllib.request
import pandas as pd
import matplotlib.pyplot as plt

URL = ('https://raw.githubusercontent.com/Nicolenki7/ETFbook-data-analytics-case-study/'
       'main/etf_flows_analysis/data/etf_flows_trend_analysis_2025.csv')
pathlib.Path('outputs').mkdir(exist_ok=True)

with urllib.request.urlopen(URL) as r:
    df = pd.read_csv(io.BytesIO(r.read()))

df['date'] = pd.to_datetime(df['date'])
df['mes'] = df['date'].dt.strftime('%Y-%m')

print('filas:', len(df), '| columnas:', list(df.columns))
print('ETFs:', sorted(df['etf_ticker'].unique()), '| meses:', sorted(df['mes'].unique()))
print('unidad: millones de USD')
df.head()

## 1. Cuanto dinero se movio

Signo incluido: un flujo neto negativo es salida de capital, no venta baja.

In [ ]:
total = df['net_flows_usd_m'].sum()
n_meses = df['mes'].nunique()

print(f'RESPUESTA: los {len(df["etf_ticker"].unique())} ETFs acumulan {total:,.0f} millones de USD en {n_meses} meses.')

mes = df.groupby('mes')['net_flows_usd_m'].sum()
print()
print(mes.to_string())

mejor = mes.idxmax()
peor = mes.idxmin()
print()
print(f'RESPUESTA: el mejor mes es {mejor} con {mes.max():,.0f} millones y el peor {peor} con {mes.min():,.0f}.')

ax = mes.plot(kind='bar', figsize=(8, 3.5), rot=0)
ax.set_ylabel('flujo neto (millones USD)')
ax.set_title('Flujo neto agregado por mes')
plt.tight_layout()
plt.savefig('outputs/flujo_mensual.png', dpi=150)
plt.show()

## 2. Quien se lleva el flujo

Cuota por ETF: con 4 fondos, la concentracion es la pregunta clave.

In [ ]:
resumen = (df.groupby('etf_ticker')['net_flows_usd_m']
             .agg(flujo_total='sum', meses='count', media='mean')
             .sort_values('flujo_total', ascending=False)
             .round(1))
resumen['cuota_pct'] = (resumen['flujo_total'] / resumen['flujo_total'].sum() * 100).round(1)
print(resumen.to_string())

primero = resumen.index[0]
print()
print(f'RESPUESTA: {primero} se lleva el {resumen["cuota_pct"].iloc[0]} % del flujo neto del periodo ({resumen["flujo_total"].iloc[0]:,.0f} millones).')

ax = resumen['flujo_total'].plot(kind='bar', figsize=(8, 3.5), rot=0)
ax.set_ylabel('flujo total (millones USD)')
ax.set_title('Flujo neto por ETF en el trimestre')
plt.tight_layout()
plt.savefig('outputs/flujo_etf.png', dpi=150)
plt.show()

## 3. Entrada sostenida o un solo mes

Un total positivo puede esconder dos meses en rojo. Miramos el signo mes a mes por ETF.

In [ ]:
pivote = df.pivot_table(index='mes', columns='etf_ticker', values='net_flows_usd_m', aggfunc='sum')
print(pivote.to_string())

meses_negativos = (pivote < 0).sum().sum()
total_celdas = pivote.size
print()
print(f'RESPUESTA: de {total_celdas} combinaciones ETF-mes, {meses_negativos} son de salida de capital.')

ax = pivote.plot(kind='bar', figsize=(9, 4), rot=0)
ax.set_ylabel('flujo neto (millones USD)')
ax.set_title('Flujo neto por ETF y mes')
ax.axhline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig('outputs/flujo_etf_mes.png', dpi=150)
plt.show()

## 4. Que region domina

La columna `region` agrupa los ETFs por mercado de origen.

In [ ]:
reg = df.groupby('region')['net_flows_usd_m'].sum().sort_values(ascending=False)
print(reg.to_string())
print()
print(f'RESPUESTA: {reg.index[0]} concentra el {reg.iloc[0] / reg.sum() * 100:.1f} % del flujo neto.')

## Limites

- 12 observaciones y 3 meses: describe el trimestre, no una tendencia.
- El flujo neto no es rentabilidad: mide entradas y salidas de dinero, no desempeno del fondo.
- La region viene dada por el dataset; no se deduce del ticker.
- Sin datos de patrimonio (AUM) no se puede relativizar el flujo sobre el tamano del fondo.